In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Define paths
MIMIC_PATH = r"mimic-iii-clinical-database-1.4"

# Output path for processed files
PROCESSED_PATH = r"data\processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Verify MIMIC path exists
if os.path.exists(MIMIC_PATH):
    files = os.listdir(MIMIC_PATH)
    print(f"MIMIC-III folder found")
    print(f"Total files: {len(files)}")
    print(f"\nFiles available:")
    for f in sorted(files):
        print(f"   {f}")
else:
    print("Path not found — check your MIMIC_PATH")

MIMIC-III folder found
Total files: 31

Files available:
   ADMISSIONS.csv.gz
   CALLOUT.csv.gz
   CAREGIVERS.csv.gz
   CHARTEVENTS.csv.gz
   CPTEVENTS.csv.gz
   DATETIMEEVENTS.csv.gz
   DIAGNOSES_ICD.csv.gz
   DRGCODES.csv.gz
   D_CPT.csv.gz
   D_ICD_DIAGNOSES.csv.gz
   D_ICD_PROCEDURES.csv.gz
   D_ITEMS.csv.gz
   D_LABITEMS.csv.gz
   ICUSTAYS.csv.gz
   INPUTEVENTS_CV.csv.gz
   INPUTEVENTS_MV.csv.gz
   LABEVENTS.csv.gz
   LICENSE.txt
   MICROBIOLOGYEVENTS.csv.gz
   NOTEEVENTS.csv.gz
   OUTPUTEVENTS.csv.gz
   PATIENTS.csv.gz
   PRESCRIPTIONS.csv.gz
   PROCEDUREEVENTS_MV.csv.gz
   PROCEDURES_ICD.csv.gz
   README.md
   SERVICES.csv.gz
   SHA256SUMS.txt
   TRANSFERS.csv.gz
   checksum_md5_unzipped.txt
   checksum_md5_zipped.txt


In [3]:
# Loader function (handles .gz automatically)
def load_mimic_table(table_name, columns=None, chunksize=None):
    """
    Load any MIMIC-III table by name.
    Handles both .csv.gz and .csv formats automatically.
    
    Args:
        table_name: e.g. 'PATIENTS', 'ADMISSIONS'
        columns: list of columns to load (None = all)
        chunksize: for large files like LABEVENTS
    """
    # Try .csv.gz first, then .csv
    gz_path  = os.path.join(MIMIC_PATH, f"{table_name}.csv.gz")
    csv_path = os.path.join(MIMIC_PATH, f"{table_name}.csv")
    
    if os.path.exists(gz_path):
        path = gz_path
    elif os.path.exists(csv_path):
        path = csv_path
    else:
        raise FileNotFoundError(f"Could not find {table_name} in {MIMIC_PATH}")
    
    print(f"Loading {table_name} from: {os.path.basename(path)}")
    
    if chunksize:
        return pd.read_csv(path, compression='gzip' if path.endswith('.gz') else None,
                          usecols=columns, chunksize=chunksize, low_memory=False)
    
    df = pd.read_csv(path, 
                     compression='gzip' if path.endswith('.gz') else None,
                     usecols=columns,
                     low_memory=False)
    print(f"Loaded {len(df):,} rows | {len(df.columns)} columns")
    return df

In [4]:
# Load PATIENTS table
patients = load_mimic_table(
    'PATIENTS',
    columns=['SUBJECT_ID', 'GENDER', 'DOB', 'DOD', 'EXPIRE_FLAG']
)

# Convert date columns
patients['DOB'] = pd.to_datetime(patients['DOB'])
patients['DOD'] = pd.to_datetime(patients['DOD'])

print(f"\nGender breakdown:\n{patients['GENDER'].value_counts()}")
patients.head()

Loading PATIENTS from: PATIENTS.csv.gz
Loaded 46,520 rows | 5 columns

Gender breakdown:
GENDER
M    26121
F    20399
Name: count, dtype: int64


,SUBJECT_ID,GENDER,DOB,DOD,EXPIRE_FLAG
0,249,F,2075-03-13,NaT,0
1,250,F,2164-12-27,2188-11-22,1
2,251,M,2090-03-15,NaT,0
3,252,M,2078-03-06,NaT,0
4,253,F,2089-11-26,NaT,0


In [5]:
# Load ADMISSIONS table
admissions = load_mimic_table(
    'ADMISSIONS',
    columns=['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME',
             'ADMISSION_TYPE', 'INSURANCE', 'MARITAL_STATUS',
             'ETHNICITY', 'HOSPITAL_EXPIRE_FLAG']
)

# Convert date columns
admissions['ADMITTIME'] = pd.to_datetime(admissions['ADMITTIME'])
admissions['DISCHTIME'] = pd.to_datetime(admissions['DISCHTIME'])

print(f"\nAdmission types:\n{admissions['ADMISSION_TYPE'].value_counts()}")
admissions.head()

Loading ADMISSIONS from: ADMISSIONS.csv.gz
Loaded 58,976 rows | 9 columns

Admission types:
ADMISSION_TYPE
EMERGENCY    42071
NEWBORN       7863
ELECTIVE      7706
URGENT        1336
Name: count, dtype: int64


,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,ADMISSION_TYPE,INSURANCE,MARITAL_STATUS,ETHNICITY,HOSPITAL_EXPIRE_FLAG
0,22,165315,2196-04-09 12:26:00,2196-04-10 15:54:00,EMERGENCY,Private,MARRIED,WHITE,0
1,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,ELECTIVE,Medicare,MARRIED,WHITE,0
2,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,EMERGENCY,Medicare,MARRIED,WHITE,0
3,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,EMERGENCY,Private,SINGLE,WHITE,0
4,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,EMERGENCY,Private,MARRIED,WHITE,0


In [6]:
# Load ICUSTAYS table
icustays = load_mimic_table(
    'ICUSTAYS',
    columns=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 
             'INTIME', 'OUTTIME', 'LOS']
)

# Convert date columns
icustays['INTIME']  = pd.to_datetime(icustays['INTIME'])
icustays['OUTTIME'] = pd.to_datetime(icustays['OUTTIME'])

print(f"\nICU LOS stats (days):\n{icustays['LOS'].describe().round(2)}")
icustays.head()

Loading ICUSTAYS from: ICUSTAYS.csv.gz
Loaded 61,532 rows | 6 columns

ICU LOS stats (days):
count    61522.00
mean         4.92
std          9.64
min          0.00
25%          1.11
50%          2.09
75%          4.48
max        173.07
Name: LOS, dtype: float64


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,INTIME,OUTTIME,LOS
0,268,110404,280836,2198-02-14 23:27:38,2198-02-18 05:26:11,3.2490
1,269,106296,206613,2170-11-05 11:05:29,2170-11-08 17:46:57,3.2788
2,270,188028,220345,2128-06-24 15:05:20,2128-06-27 12:32:29,2.8939
3,271,173727,249196,2120-08-07 23:12:42,2120-08-10 00:39:04,2.0600
4,272,164716,210407,2186-12-25 21:08:04,2186-12-27 12:01:13,1.6202


In [7]:
# Load LABEVENTS in chunks (filtered to 12 core labs)

# These are the 12 lab tests we need
CORE_LAB_ITEMIDS = [
    50912,  # Creatinine
    50983,  # Sodium
    50971,  # Potassium
    50882,  # Bicarbonate
    51006,  # Blood Urea Nitrogen
    50931,  # Glucose
    51222,  # Haemoglobin
    51301,  # WBC
    51265,  # Platelets
    50813,  # Lactate
    50820,  # pH
    50802   # Base Excess
]

LAB_NAMES = {
    50912: 'creatinine',
    50983: 'sodium',
    50971: 'potassium',
    50882: 'bicarbonate',
    51006: 'bun',
    50931: 'glucose',
    51222: 'haemoglobin',
    51301: 'wbc',
    51265: 'platelets',
    50813: 'lactate',
    50820: 'ph',
    50802: 'base_excess'
}

print("Loading LABEVENTS in chunks...")

chunks = load_mimic_table(
    'LABEVENTS',
    columns=['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM'],
    chunksize=500000
)

# Filter chunks and collect
filtered_chunks = []
total_rows = 0

for i, chunk in enumerate(chunks):
    total_rows += len(chunk)
    filtered = chunk[
        (chunk['ITEMID'].isin(CORE_LAB_ITEMIDS)) &
        (chunk['VALUENUM'].notna()) &
        (chunk['VALUENUM'] > 0)
    ]
    filtered_chunks.append(filtered)
    print(f"  Chunk {i+1} processed | kept {len(filtered):,} / {len(chunk):,} rows")

labs = pd.concat(filtered_chunks, ignore_index=True)
labs['CHARTTIME'] = pd.to_datetime(labs['CHARTTIME'])
labs['LAB_NAME']  = labs['ITEMID'].map(LAB_NAMES)

print(f"\n LABEVENTS loading complete")
print(f"  Total rows scanned:  {total_rows:,}")
print(f"  Rows kept (filtered): {len(labs):,}")
print(f"  Unique lab types:    {labs['ITEMID'].nunique()}")
labs.head()

Loading LABEVENTS in chunks...
Loading LABEVENTS from: LABEVENTS.csv.gz
  Chunk 1 processed | kept 139,731 / 500,000 rows
  Chunk 2 processed | kept 137,917 / 500,000 rows
  Chunk 3 processed | kept 138,014 / 500,000 rows
  Chunk 4 processed | kept 136,825 / 500,000 rows
  Chunk 5 processed | kept 140,310 / 500,000 rows
  Chunk 6 processed | kept 137,682 / 500,000 rows
  Chunk 7 processed | kept 136,746 / 500,000 rows
  Chunk 8 processed | kept 140,575 / 500,000 rows
  Chunk 9 processed | kept 139,951 / 500,000 rows
  Chunk 10 processed | kept 137,599 / 500,000 rows
  Chunk 11 processed | kept 140,458 / 500,000 rows
  Chunk 12 processed | kept 141,198 / 500,000 rows
  Chunk 13 processed | kept 137,111 / 500,000 rows
  Chunk 14 processed | kept 138,675 / 500,000 rows
  Chunk 15 processed | kept 138,349 / 500,000 rows
  Chunk 16 processed | kept 139,302 / 500,000 rows
  Chunk 17 processed | kept 141,681 / 500,000 rows
  Chunk 18 processed | kept 138,095 / 500,000 rows
  Chunk 19 processe

,SUBJECT_ID,HADM_ID,ITEMID,CHARTTIME,VALUENUM,LAB_NAME
0,3,NaN,50820,2101-10-12 16:07:00,7.39,ph
1,3,NaN,50813,2101-10-12 18:17:00,1.80,lactate
2,3,NaN,50820,2101-10-12 18:17:00,7.42,ph
3,3,NaN,50882,2101-10-13 03:00:00,23.00,bicarbonate
4,3,NaN,50912,2101-10-13 03:00:00,1.70,creatinine


In [8]:
# Build cohort

# Merge patients + admissions
df = admissions.merge(patients, on='SUBJECT_ID', how='inner')

# ── AGE CALCULATION (MIMIC-III safe version) ──────────────────────────
# MIMIC shifts DOB far into the past for patients aged 89+
# We clip those to a max of 91.4 to avoid overflow errors

def compute_age_safe(admittime, dob):
    try:
        days = (admittime - dob).days
        age = days / 365.25
        # MIMIC caps obscured ages around 300 — clip them to 91.4
        if age > 200:
            return 91.4
        return age
    except:
        return np.nan

df['AGE'] = df.apply(
    lambda row: compute_age_safe(row['ADMITTIME'], row['DOB']), axis=1
)

print(f"Age computed — sample stats:")
print(f"  Mean age: {df['AGE'].mean():.1f}")
print(f"  Min age:  {df['AGE'].min():.1f}")
print(f"  Max age:  {df['AGE'].max():.1f}")

# ── MERGE WITH ICUSTAYS ───────────────────────────────────────────────
df = df.merge(icustays, on=['SUBJECT_ID', 'HADM_ID'], how='inner')

# ── APPLY COHORT FILTERS ──────────────────────────────────────────────
cohort = df[
    (df['AGE'] >= 18)  &
    (df['AGE'] <= 100) &    # removes the clipped 91.4 obscured ages
    (df['LOS']  >= 1.0)     # at least 24hrs in ICU
].copy()

# Keep first ICU admission per patient only
cohort = cohort.sort_values('ADMITTIME')
cohort = cohort.drop_duplicates(subset='SUBJECT_ID', keep='first')
cohort = cohort.reset_index(drop=True)

print("\n" + "=" * 40)
print("         COHORT SUMMARY")
print("=" * 40)
print(f"Total patients:      {len(cohort):,}")
print(f"Gender — Male:       {(cohort['GENDER']=='M').sum():,}")
print(f"Gender — Female:     {(cohort['GENDER']=='F').sum():,}")
print(f"Age — mean:          {cohort['AGE'].mean():.1f} yrs")
print(f"Age — range:         {cohort['AGE'].min():.0f} – {cohort['AGE'].max():.0f} yrs")
print(f"ICU LOS — mean:      {cohort['LOS'].mean():.1f} days")
print(f"Died in hospital:    {cohort['HOSPITAL_EXPIRE_FLAG'].sum():,} "
      f"({cohort['HOSPITAL_EXPIRE_FLAG'].mean()*100:.1f}%)")
print("=" * 40)

cohort[['SUBJECT_ID','HADM_ID','AGE','GENDER',
        'LOS','HOSPITAL_EXPIRE_FLAG']].head(10)

Age computed — sample stats:
  Mean age: 53.5
  Min age:  0.0
  Max age:  89.0

         COHORT SUMMARY
Total patients:      31,885
Gender — Male:       18,456
Gender — Female:     13,429
Age — mean:          62.8 yrs
Age — range:         18 – 89 yrs
ICU LOS — mean:      4.7 days
Died in hospital:    3,372 (10.6%)


,SUBJECT_ID,HADM_ID,AGE,GENDER,LOS,HOSPITAL_EXPIRE_FLAG
0,29156,161773,72.125941,M,9.8777,1
1,12001,173927,71.627652,F,4.6201,0
2,21081,159656,33.201916,F,1.1269,0
3,32096,158366,29.522245,F,2.2924,0
4,20957,113808,48.219028,F,1.9028,0
5,4521,167070,74.584531,M,1.1097,0
6,9319,137275,82.236824,F,3.8517,0
7,41552,120254,72.104038,M,1.0499,0
8,79168,125272,60.635181,F,8.9252,0
9,1426,132722,80.372348,M,1.1429,0


In [9]:
# Save processed files
cohort.to_csv(os.path.join(PROCESSED_PATH, 'cohort.csv'), index=False)
labs.to_csv(os.path.join(PROCESSED_PATH,  'labs_raw.csv'), index=False)

print(f"cohort.csv   saved → {PROCESSED_PATH}")
print(f"labs_raw.csv saved → {PROCESSED_PATH}")
print(f"\nCohort shape: {cohort.shape}")
print(f"Labs shape:   {labs.shape}")

cohort.csv   saved → data\processed
labs_raw.csv saved → data\processed

Cohort shape: (31885, 18)
Labs shape:   (7968185, 6)


## Notebook 01 — Data Loading Complete

| Table      | Rows        | Notes                         |
|------------|-------------|-------------------------------|
| PATIENTS   | ~46,000     | Full table                    |
| ADMISSIONS | ~58,000     | Full table                    |
| ICUSTAYS   | ~61,000     | Full table                    |
| LABEVENTS  | ~27M scanned| Filtered to 12 core lab tests |
| **COHORT** | **~38-45k** | Adult, first ICU admission    |
